# Symmetry-sector reduction

This notebook compares the same translation-invariant spin model in the full Hilbert space and in symmetry-reduced sectors. It also checks that splitting a sector by parity reproduces the same spectrum when recombined.


In [ ]:
using LinearAlgebra
DEV = true

function find_local_edkit(start = pwd())
    dir = abspath(start)
    while true
        candidate = joinpath(dir, "src", "EDKit.jl")
        isfile(candidate) && return candidate
        parent = dirname(dir)
        parent == dir && error("Could not locate src/EDKit.jl from $(start)")
        dir = parent
    end
end

if DEV
    include(find_local_edkit())
    using .EDKit
else
    using EDKit
end


In [ ]:
L = 8
bond = spin((1.0, "xx"), (1.0, "yy"))

full_basis = TensorBasis(L = L, base = 2)
nk_basis = basis(L = L, N = L ÷ 2, k = 0)
even_basis = basis(L = L, N = L ÷ 2, k = 0, p = 1)
odd_basis = basis(L = L, N = L ÷ 2, k = 0, p = -1)

full_H = trans_inv_operator(bond, 2, full_basis)
nk_H = trans_inv_operator(bond, 2, nk_basis)
even_H = trans_inv_operator(bond, 2, even_basis)
odd_H = trans_inv_operator(bond, 2, odd_basis)


In [ ]:
nk_vals = eigvals(Hermitian(Array(nk_H))) |> sort
split_vals = vcat(
    eigvals(Hermitian(Array(even_H))),
    eigvals(Hermitian(Array(odd_H))),
) |> sort

summary = (; 
    full_dimension = size(full_basis, 1),
    nk_dimension = size(nk_basis, 1),
    even_dimension = size(even_basis, 1),
    odd_dimension = size(odd_basis, 1),
    parity_recombination_error = norm(nk_vals - split_vals),
    lowest_five_sector_energies = nk_vals[1:5],
)

summary
